# 02 — Geolocate IP Addresses from a CSV with IP2Location.io

**Use case:** Enrich a CSV containing IP addresses with country, region, city, coordinates, time zone, ASN, and basic proxy information. Export a new CSV with the original rows and lookup results.

**What you will learn:**
1. Read and preview a CSV with `pandas`.
2. Validate IPv4 and IPv6 addresses locally, and skip private/invalid values.
3. Deduplicate IPs to minimize API calls while retaining duplicate input rows.
4. Query the [IP2Location.io IP Geolocation API](https://www.ip2location.io/ip2location-documentation) with optional authentication.
5. Handle network errors, API errors, and depleted quotas without discarding earlier results.
6. Export a CSV suitable for further analysis.

**Note:** This notebook uses the **standard single-IP endpoint**, once per unique public IP. It does *not* require access to the separately documented bulk API. An internet connection is required for live lookups.

## 1. Install packages

Run this in a Jupyter notebook. `%pip` installs packages into the active notebook environment.

In [ ]:
%pip install -q requests pandas

## 2. Configure paths and API access

- Use the supplied `sample_ips.csv`, or set `INPUT_CSV` to your own CSV with an **`ip`** column. Other columns (such as `label`) will be preserved.
- Leave `IP2LOCATION_API_KEY` unset for **keyless requests**, or set the environment variable before starting Jupyter. The key is sent in an `Authorization: Bearer` header rather than written into a URL or output file.
- The default `MAX_UNIQUE_LOOKUPS=25` protects against accidentally processing a very large file. Raise it deliberately when needed.

See [documentation](https://www.ip2location.io/ip2location-documentation) for current limits and field availability; keyless access currently allows **1,000 lookups/day**, and the Free Plan with an API key allows **50,000 lookups/month**. Each unique public IP queried uses an API request.

In [ ]:
import ipaddress
import os
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

API_URL = "https://api.ip2location.io/"
API_KEY = os.environ.get("IP2LOCATION_API_KEY", "").strip()
INPUT_CSV = Path("sample_ips.csv")
OUTPUT_CSV = Path("ip_geolocation_results.csv")
MAX_UNIQUE_LOOKUPS = 25  # Explicit safety limit; adjust for your use case.
REQUEST_DELAY_SECONDS = 0.25  # Polite pacing between successive requests.
MAX_RETRIES = 2  # Retry transient connection/server failures only.

assert MAX_UNIQUE_LOOKUPS >= 0
assert REQUEST_DELAY_SECONDS >= 0
assert MAX_RETRIES >= 0
print(f"Input: {INPUT_CSV} | Output: {OUTPUT_CSV}")
print(f"API key configured: {'yes' if API_KEY else 'no (keyless mode)'}")

## 3. Create demo data (only if the input file is missing)

The download includes `sample_ips.csv`. If you run the notebook on its own without downloading that file, this cell also creates a demonstration CSV. It includes repeated, IPv6, private, invalid, and empty IP values to show how the workflow handles them. **Existing files are never overwritten by this cell.**

In [ ]:
import csv

if not INPUT_CSV.exists():
    demo_rows = [
        ("8.8.8.8", "Google DNS"),
        ("1.1.1.1", "Cloudflare DNS"),
        ("9.9.9.9", "Quad9 DNS"),
        ("208.67.222.222", "OpenDNS"),
        ("2606:4700:4700::1111", "Cloudflare IPv6 DNS"),
        ("8.8.8.8", "Duplicate: Google DNS"),
        ("192.168.1.1", "Private IP: should skip"),
        ("not-an-ip", "Invalid value: should skip"),
        ("", "Blank value: should skip"),
    ]
    with INPUT_CSV.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["ip", "label"])
        writer.writerows(demo_rows)
    print(f"Created demo input: {INPUT_CSV}")
else:
    print(f"Using existing input file: {INPUT_CSV}")

## 4. Read the CSV

Read all CSV columns as strings so that values do not get unintentionally converted into numbers. Every row from the original CSV will appear in the final output, including invalid IPs.

In [ ]:
input_df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
if "ip" not in input_df.columns:
    raise ValueError("Input CSV needs a column named 'ip'.")

print(f"Loaded {len(input_df):,} input rows")
display(input_df.head(12))

## 5. Validate, normalize and deduplicate IP addresses

Only globally routable addresses are sent to the API. Invalid, empty and non-global (such as RFC1918 private) addresses are marked but never queried. Normalization also prevents the same IPv6 IP written in different forms from consuming extra requests.

In [ ]:
def prepare_ip(value: str) -> tuple[str, str, str]:
    """Return (normalized_ip, input_status, reason)."""
    raw = str(value).strip()
    if not raw:
        return "", "missing_ip", "Empty IP address"
    try:
        parsed = ipaddress.ip_address(raw)
    except ValueError:
        return "", "invalid_ip", "Not a valid IPv4 or IPv6 address"
    if not parsed.is_global:
        return str(parsed), "non_public_ip", "Not globally routable"
    return str(parsed), "ready", ""

prepared = input_df["ip"].map(prepare_ip)
input_df = input_df.copy()
input_df["normalized_ip"] = prepared.map(lambda values: values[0])
input_df["input_status"] = prepared.map(lambda values: values[1])
input_df["input_error"] = prepared.map(lambda values: values[2])

# dict.fromkeys preserves order while removing duplicates.
unique_ips = list(dict.fromkeys(
    input_df.loc[input_df["input_status"] == "ready", "normalized_ip"]
))
lookup_ips = unique_ips[:MAX_UNIQUE_LOOKUPS]

print(f"Input rows: {len(input_df):,}")
print(f"Unique public IPs: {len(unique_ips):,}")
print(f"API requests planned: {len(lookup_ips):,}")
print(f"Skipped by safety cap: {len(unique_ips) - len(lookup_ips):,}")
display(input_df[["ip", "normalized_ip", "input_status", "input_error"]].head(12))

## 6. Define a robust API lookup function

The function reuses an HTTP session, enforces a timeout, and retries temporary network or server failures up to `MAX_RETRIES`. It stops the batch on HTTP 401/403/429 or API error `10000` (invalid credentials or insufficient queries). Plan-dependent fields may be missing from successful responses, so they are extracted with `.get()`.

In [ ]:
FIELDS = [
    "country_code", "country_name", "region_name", "city_name",
    "latitude", "longitude", "zip_code", "time_zone", "asn", "as", "is_proxy",
]

class QuotaOrAuthError(RuntimeError):
    """Stop the batch when the API key/quota/rate limit prevents lookups."""


def lookup_one_ip(session: requests.Session, ip: str) -> dict:
    """Return a status + selected geolocation fields for one public IP."""
    for attempt in range(MAX_RETRIES + 1):
        try:
            response = session.get(
                API_URL,
                params={"ip": ip, "format": "json"},
                timeout=(5, 15),
            )
            if response.status_code in (401, 403, 429):
                raise QuotaOrAuthError(f"HTTP {response.status_code}: check API access or quota")
            if 500 <= response.status_code < 600 and attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)
                continue
            response.raise_for_status()
            payload = response.json()

            if not isinstance(payload, dict):
                return {"status": "response_error", "error": "Expected a JSON object"}
            if payload.get("error"):
                api_error = payload["error"]
                error_code = str(api_error.get("error_code", "")) if isinstance(api_error, dict) else ""
                error_message = (api_error.get("error_message", "API error")
                                 if isinstance(api_error, dict) else str(api_error))
                if error_code == "10000":
                    raise QuotaOrAuthError(f"API error {error_code}: {error_message}")
                return {"status": "api_error", "error": f"{error_code}: {error_message}"}

            return {
                "status": "ok",
                "error": "",
                **{field: payload.get(field) for field in FIELDS},
            }
        except (requests.Timeout, requests.ConnectionError) as exc:
            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)
                continue
            return {"status": "network_error", "error": type(exc).__name__}
        except requests.HTTPError as exc:
            code = exc.response.status_code if exc.response is not None else "unknown"
            return {"status": "http_error", "error": f"HTTP {code}"}
        except requests.RequestException as exc:
            return {"status": "network_error", "error": type(exc).__name__}
        except ValueError:
            return {"status": "response_error", "error": "Invalid JSON response"}

    return {"status": "response_error", "error": "No response"}

## 7. Perform the lookups

Each **distinct public IP** is requested at most once in this run, even if it appears many times in the input CSV. A quota or authentication failure stops subsequent requests, but earlier results remain available. The sample uses only a handful of queries.

In [ ]:
lookup_results = {
    ip: {"status": "skipped_limit", "error": "MAX_UNIQUE_LOOKUPS safety cap"}
    for ip in unique_ips[MAX_UNIQUE_LOOKUPS:]
}

with requests.Session() as session:
    if API_KEY:
        session.headers.update({"Authorization": f"Bearer {API_KEY}"})
    for index, ip in enumerate(lookup_ips):
        try:
            lookup_results[ip] = lookup_one_ip(session, ip)
        except QuotaOrAuthError as exc:
            lookup_results[ip] = {"status": "quota_or_auth_error", "error": str(exc)}
            for remaining_ip in lookup_ips[index + 1:]:
                lookup_results[remaining_ip] = {
                    "status": "skipped_quota", "error": "Batch stopped after quota/auth error"
                }
            print(f"Stopping API calls: {exc}")
            break
        print(f"[{index + 1}/{len(lookup_ips)}] {ip}: {lookup_results[ip]['status']}")
        if index + 1 < len(lookup_ips):
            time.sleep(REQUEST_DELAY_SECONDS)

print(f"Unique IP results collected: {len(lookup_results):,}")

## 8. Join results to **every original CSV row**

This step maps each normalized IP back to the original rows. Duplicate IPs retain their separate `label` values, and malformed/non-public inputs receive an explicit `status` and `error`. Empty values for some geolocation fields are expected on lower-tier plans.

In [ ]:
output_rows = []
for source_row in input_df.to_dict(orient="records"):
    enriched = dict(source_row)
    if source_row["input_status"] == "ready":
        result = lookup_results.get(source_row["normalized_ip"], {
            "status": "skipped", "error": "Not queried"
        })
    else:
        result = {
            "status": source_row["input_status"],
            "error": source_row["input_error"],
        }
    enriched["status"] = result["status"]
    enriched["error"] = result.get("error", "")
    for field in FIELDS:
        enriched[field] = result.get(field)
    output_rows.append(enriched)

output_columns = list(input_df.columns) + ["status", "error"] + FIELDS
enriched_df = pd.DataFrame(output_rows, columns=output_columns)
print("Status counts:")
print(enriched_df["status"].value_counts(dropna=False).to_string())
display(enriched_df.head(15))

## 9. Export results to a CSV

The file `ip_geolocation_results.csv` will be saved in the current notebook working directory. Running this export cell again overwrites the *output* file, never the input file.

In [ ]:
enriched_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved {len(enriched_df):,} rows to: {OUTPUT_CSV.resolve()}")

# Optional: check that the exported CSV can be read back.
display(pd.read_csv(OUTPUT_CSV).head(10))

## 10. Try your own CSV

1. Replace `sample_ips.csv` with your CSV file, or change `INPUT_CSV` in Section 2. The **`ip`** column is required; all other columns are optional.
2. Increase `MAX_UNIQUE_LOOKUPS` only after reviewing the planned number of requests in Section 5.
3. Set `IP2LOCATION_API_KEY` in the environment to use an authenticated plan.
4. Run Sections 4–9 again to write a fresh result file.
5. Explore the results, for example with `enriched_df["country_code"].value_counts()`.

**Notes:**
- Geolocation is approximate and data fields vary by subscription plan.
- Free-plan and keyless users are required to give visible attribution when using the service publicly. See [IP2Location.io pricing and attribution details](https://www.ip2location.io/pricing).
- This notebook shows CSV enrichment using repeated single-IP requests. For large paid-plan workflows, see the [separate bulk API documentation](https://www.ip2location.io/documentation).
- Avoid committing API keys or sensitive original input data to public repositories.